# CALE Cross-Framing Validity Visualization

This notebook is for the framing/stress follow-up experiment. It reads multiple `*_eval_report.json` files, merges their row-level predictions, and visualizes construct-validity evidence across framing conditions.

The goal is not FEVER label accuracy. The goal is to test whether evaluator variants behave consistently and construct-sensitively when the same factual resource is presented with different adversarial framings.

In [ ]:
from pathlib import Path
import json
import re
import textwrap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 220,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "font.size": 10,
})

FRAMING_ORDER = ["neutral", "assertive", "authoritative", "polite_misleading"]
VARIANT_ORDER = [
    "baseline_binary",
    "baseline_likert",
    "direct_trustllm_heuristic",
    "generic_cale",
    "attack_aware_cale",
    "full_attack_aware_cale",
]
LABEL_ORDER = ["not_corrected", "uncertain", "partially_corrected", "corrected"]
DIAGNOSTIC_METRICS = ["source_faithfulness_rate", "misinformation_detection_rate", "framing_resistance_rate"]

# Leave empty to auto-discover. Fill manually if your reports live elsewhere.
REPORT_PATHS = []

AUTO_REPORT_GLOBS = [
    "outputs/*fever_dev*neutral*_eval_report.json",
    "outputs/*fever_dev*assertive*_eval_report.json",
    "outputs/*fever_dev*authoritative*_eval_report.json",
    "outputs/*fever_dev*polite_misleading*_eval_report.json",
    "outputs/small_models_all/*fever_dev*neutral*_eval_report.json",
    "outputs/small_models_all/*fever_dev*assertive*_eval_report.json",
    "outputs/small_models_all/*fever_dev*authoritative*_eval_report.json",
    "outputs/small_models_all/*fever_dev*polite_misleading*_eval_report.json",
]

def infer_framing_from_path(path):
    name = Path(path).name
    for framing in sorted(FRAMING_ORDER, key=len, reverse=True):
        if f"_{framing}_" in name or framing in name:
            return framing
    return "unknown"

def nice_name(value):
    return str(value).replace("Qwen/", "").replace("meta-llama/", "")

def nice_ylim(values, metric="score", min_span=0.04):
    values = pd.Series(values).dropna().astype(float)
    if values.empty:
        return (0, 1)
    ymin, ymax = values.min(), values.max()
    span = max(ymax - ymin, min_span)
    pad = span * 0.35
    lower, upper = ymin - pad, ymax + pad
    if metric not in {"delta", "score_shift"}:
        lower, upper = max(0, lower), min(1, upper)
    if upper <= lower:
        upper = lower + min_span
    return lower, upper

AUTO_DISCOVERED_REPORTS = not REPORT_PATHS

if not REPORT_PATHS:
    paths = []
    for pattern in AUTO_REPORT_GLOBS:
        paths.extend(Path(".").glob(pattern))
    REPORT_PATHS = sorted({str(path) for path in paths})

if not REPORT_PATHS:
    raise FileNotFoundError(
        "No report JSON files found. Fill REPORT_PATHS with paths like "
        "outputs/fever_dev_qwen25_15b_llama32_1b_assertive_smoke_eval_report.json"
    )

reports = []
pred_frames = []
metric_frames = []

for path in REPORT_PATHS:
    path = Path(path)
    report = json.loads(path.read_text(encoding="utf-8"))
    framing = infer_framing_from_path(path)
    run_config = report.get("run_config", {})
    rows = report.get("predictions", [])
    metrics = report.get("metrics", {})
    reports.append({"path": str(path), "framing": framing, "n_predictions": len(rows), **run_config})
    if rows:
        df = pd.DataFrame(rows)
        df["report_path"] = str(path)
        df["framing"] = framing
        df["model_short"] = df.get("model_name", "unknown").map(nice_name)
        pred_frames.append(df)
    for variant, values in metrics.items():
        row = {"report_path": str(path), "framing": framing, "variant": variant}
        row.update(values)
        metric_frames.append(row)

run_df_all = pd.DataFrame(reports)
pred_df_all = pd.concat(pred_frames, ignore_index=True) if pred_frames else pd.DataFrame()
metrics_df_all = pd.DataFrame(metric_frames)

# Cross-framing comparisons must use reports from the same run scale.
# A common situation is that the folder contains both:
#   1. smoke framing reports, e.g. 500 items x 2 models x 6 variants = 6,000 predictions;
#   2. a full neutral report, e.g. 19,998 items x 2 models x 6 variants = 239,976 predictions.
# Mixing them would make framing plots look like a framing effect when it is really a sample-size effect.
# Therefore, auto-discovery keeps the prediction-count group that covers the most framing conditions
# and drops mismatched groups by default. The dropped reports are still on disk; they are just not used here.
selected_report_paths = set(run_df_all["path"]) if not run_df_all.empty else set()
if AUTO_DISCOVERED_REPORTS and not run_df_all.empty and run_df_all["n_predictions"].nunique() > 1:
    candidates = (
        run_df_all.groupby("n_predictions")
        .agg(
            n_reports=("path", "count"),
            n_framings=("framing", "nunique"),
            framings=("framing", lambda values: ", ".join(sorted(set(map(str, values))))),
        )
        .reset_index()
        .sort_values(["n_framings", "n_predictions"], ascending=[False, False])
    )
    selected_n = int(candidates.iloc[0]["n_predictions"])
    selected_report_paths = set(run_df_all.loc[run_df_all["n_predictions"] == selected_n, "path"])
    dropped = sorted(set(run_df_all["path"]) - selected_report_paths)
    print("Auto-discovery found reports from multiple run scales.")
    print("For cross-framing validity, the notebook keeps one balanced prediction-count group by default.")
    print("Candidate groups:")
    for _, row in candidates.iterrows():
        marker = "<-- kept" if int(row["n_predictions"]) == selected_n else "<-- dropped"
        print(
            f"  n_predictions={int(row['n_predictions']):,} | reports={int(row['n_reports'])} | "
            f"framings={row['framings']} {marker}"
        )
    print(
        f"Kept n_predictions={selected_n:,} because it covers the most framing conditions. "
        f"Dropped {len(dropped)} mismatched report(s) to avoid mixing smoke and full runs."
    )
    for path in dropped:
        print("  dropped:", path)

run_df = run_df_all[run_df_all["path"].isin(selected_report_paths)].copy()
pred_df = pred_df_all[pred_df_all["report_path"].isin(selected_report_paths)].copy() if not pred_df_all.empty else pred_df_all
metrics_df = metrics_df_all[metrics_df_all["report_path"].isin(selected_report_paths)].copy() if not metrics_df_all.empty else metrics_df_all
REPORT_PATHS = sorted(selected_report_paths)

OUTDIR = Path("figures") / "framing_validity"
OUTDIR.mkdir(parents=True, exist_ok=True)

print("Reports loaded:", len(REPORT_PATHS))
print("Prediction rows:", f"{len(pred_df):,}")
display(run_df[[c for c in ["framing", "path", "n_predictions", "dataset_path", "target_models", "variants"] if c in run_df.columns]])

## Metric Interpretation

Use shared outcome metrics for baseline-vs-CALE comparisons. Use CALE-only diagnostic metrics to explain what additional construct-level information CALE exposes.

In [ ]:
metric_catalog = pd.DataFrame([
    ["mean_score", "Mean final evaluator score", "mean(predictions.score)", "baseline + CALE", "Main shared outcome comparison"],
    ["label_distribution", "Distribution of corrected / partially_corrected / uncertain / not_corrected", "normalized count(predictions.label)", "baseline + CALE", "Shows judgment style and caution"],
    ["NEI overclaim rate", "Share of NEI cases judged corrected or partially_corrected", "mean(label in corrected/partially_corrected | reference_label=NEI)", "baseline + CALE", "Boundary behavior; lower is better"],
    ["score shift from neutral", "How much scores move when framing changes", "score_framing - score_neutral", "baseline + CALE", "Framing stability / discriminant validity"],
    ["label flip from neutral", "Whether final evaluator label changes under framing", "label_framing != label_neutral", "baseline + CALE", "Framing robustness; lower is better when factual core is fixed"],
    ["source_faithfulness_rate", "Average CALE Source Faithfulness subscore", "mean(subscores['Source Faithfulness'])", "CALE only", "Diagnostic interpretability"],
    ["misinformation_detection_rate", "Average CALE Misinformation Detection subscore", "mean(subscores['Misinformation Detection'])", "CALE only", "Diagnostic interpretability"],
    ["framing_resistance_rate", "Average CALE Framing Resistance subscore", "mean(subscores['Framing Resistance'])", "CALE only", "Diagnostic interpretability"],
], columns=["metric", "meaning", "computed_from", "comparable_variants", "recommended_use"])
display(metric_catalog)

## Shared Outcome Metrics Across Framing

**What is computed.** For each `framing × evaluator variant`, this section computes the mean final evaluator score and the NEI overclaim rate. `mean_score = mean(predictions.score)`. `NEI overclaim rate = share of NOT ENOUGH INFO cases judged as corrected or partially_corrected`.

**Why it matters for the paper.** These are shared outcome metrics, so they are fair for baseline-vs-CALE comparison. If CALE is construct-aligned, it should avoid giving inflated scores merely because a prompt is assertive or authoritative, and it should avoid rewarding unsupported NEI overclaims.

In [ ]:
if pred_df.empty:
    raise ValueError("Row-level predictions are required for shared outcome plots.")

summary = (
    pred_df.groupby(["framing", "variant"], dropna=False)
    .agg(n=("id", "count"), mean_score=("score", "mean"), mean_uncertainty=("uncertainty", "mean"))
    .reset_index()
)

if "reference_label" in pred_df.columns:
    nei = pred_df[pred_df["reference_label"] == "NOT ENOUGH INFO"].copy()
    if not nei.empty:
        nei["overclaim"] = nei["label"].isin(["corrected", "partially_corrected"]).astype(float)
        nei_summary = nei.groupby(["framing", "variant"])["overclaim"].mean().rename("nei_overclaim_rate").reset_index()
        summary = summary.merge(nei_summary, on=["framing", "variant"], how="left")

summary["framing"] = pd.Categorical(summary["framing"], categories=FRAMING_ORDER, ordered=True)
summary["variant"] = pd.Categorical(summary["variant"], categories=VARIANT_ORDER, ordered=True)
summary = summary.sort_values(["variant", "framing"])
display(summary)

def plot_metric_by_framing(metric, title, ylabel, filename, lower_is_better=False):
    if metric not in summary.columns or summary[metric].dropna().empty:
        print(f"Skipping {metric}: no values.")
        return
    fig, ax = plt.subplots(figsize=(11, 5))
    for variant, sub in summary.dropna(subset=[metric]).groupby("variant", observed=True):
        sub = sub.sort_values("framing")
        ax.plot(sub["framing"].astype(str), sub[metric], marker="o", linewidth=2, label=str(variant))
        for _, row in sub.iterrows():
            ax.annotate(f"{row[metric]:.3f}", (str(row["framing"]), row[metric]), textcoords="offset points", xytext=(0, 5), ha="center", fontsize=8)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("framing")
    ax.set_ylim(*nice_ylim(summary[metric], metric))
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)
    fig.tight_layout()
    fig.savefig(OUTDIR / filename)

plot_metric_by_framing("mean_score", "Mean Evaluator Score Across Framing", "mean score", "framing_mean_score_lines.png")
plot_metric_by_framing("nei_overclaim_rate", "NEI Overclaim Rate Across Framing (lower is better)", "NEI overclaim rate", "framing_nei_overclaim_lines.png", lower_is_better=True)

## Label Distribution Across Framing

**What is computed.** For each `framing × evaluator variant`, this section normalizes the counts of final evaluator labels: `not_corrected`, `uncertain`, `partially_corrected`, and `corrected`.

**Why it matters for the paper.** Label distributions show the evaluator's decision style. A direct baseline may collapse many cases into `corrected`, while CALE may expose more `uncertain` or `partially_corrected` outcomes when the construct requires caution. Each stacked segment is labeled with its share when large enough to read.

In [ ]:
label_counts = pred_df.groupby(["framing", "variant", "label"]).size().rename("n").reset_index()
label_counts["share"] = label_counts["n"] / label_counts.groupby(["framing", "variant"])["n"].transform("sum")

for variant in [v for v in VARIANT_ORDER if v in set(label_counts["variant"])] :
    sub = label_counts[label_counts["variant"] == variant]
    pivot = sub.pivot(index="framing", columns="label", values="share").fillna(0)
    pivot = pivot.reindex(FRAMING_ORDER).dropna(how="all")
    keep = [label for label in LABEL_ORDER if label in pivot.columns]
    if not keep:
        continue
    ax = pivot[keep].plot(kind="bar", stacked=True, figsize=(9, 4.4), colormap="Set2")
    ax.set_title(f"Final Label Distribution Across Framing: {variant}")
    ax.set_ylabel("share")
    ax.set_xlabel("framing")
    ax.legend(title="label", bbox_to_anchor=(1.02, 1), loc="upper left")
    for container in ax.containers:
        labels = [f"{bar.get_height():.2f}" if bar.get_height() >= 0.04 else "" for bar in container]
        ax.bar_label(container, labels=labels, label_type="center", fontsize=8, color="#111827")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(OUTDIR / f"label_distribution_by_framing_{variant}.png")

## Cross-Framing Stability Relative To Neutral

**What is computed.** This section matches rows by `id × model_name × variant`, treats `neutral` as the reference condition, and computes `score_shift = score_framing - score_neutral`, `mean_abs_score_shift`, and `label_flip_rate = mean(label_framing != label_neutral)`.

**Why it matters for the paper.** This is the key discriminant-validity test. If the factual core is fixed, evaluator judgments should not swing wildly only because the prompt is more assertive or authoritative. Lower label-flip rate and smaller score shifts support framing robustness.

In [ ]:
key_cols = ["id", "model_name", "variant"]
if not set(key_cols).issubset(pred_df.columns):
    raise ValueError(f"Predictions missing required keys for matched framing analysis: {key_cols}")

neutral = pred_df[pred_df["framing"] == "neutral"][[*key_cols, "score", "label"]].rename(columns={"score": "neutral_score", "label": "neutral_label"})
non_neutral = pred_df[pred_df["framing"] != "neutral"][[*key_cols, "framing", "score", "label", "reference_label"] if "reference_label" in pred_df.columns else [*key_cols, "framing", "score", "label"]]
matched = non_neutral.merge(neutral, on=key_cols, how="inner")

if matched.empty:
    print("No matched neutral/non-neutral rows found yet. Run neutral plus at least one other framing with matching ids.")
    stability_df = pd.DataFrame()
else:
    matched["score_shift"] = matched["score"] - matched["neutral_score"]
    matched["abs_score_shift"] = matched["score_shift"].abs()
    matched["label_flip"] = matched["label"] != matched["neutral_label"]
    stability_df = (
        matched.groupby(["framing", "variant"], dropna=False)
        .agg(
            n=("id", "count"),
            mean_score_shift=("score_shift", "mean"),
            mean_abs_score_shift=("abs_score_shift", "mean"),
            label_flip_rate=("label_flip", "mean"),
        )
        .reset_index()
    )
    stability_df["framing"] = pd.Categorical(stability_df["framing"], categories=FRAMING_ORDER, ordered=True)
    stability_df["variant"] = pd.Categorical(stability_df["variant"], categories=VARIANT_ORDER, ordered=True)
    display(stability_df.sort_values(["variant", "framing"]))

def heatmap(df, index, columns, values, title, filename, vmin=None, vmax=None, cmap="viridis"):
    if df.empty or values not in df.columns:
        print(f"Skipping heatmap {title}: no data.")
        return
    pivot = df.pivot_table(index=index, columns=columns, values=values, aggfunc="mean")
    if index == "variant":
        pivot = pivot.reindex([v for v in VARIANT_ORDER if v in pivot.index])
    if columns == "framing":
        pivot = pivot[[c for c in FRAMING_ORDER if c in pivot.columns]]
    fig, ax = plt.subplots(figsize=(max(7, 1.8 * len(pivot.columns)), max(4, 0.5 * len(pivot))))
    im = ax.imshow(pivot.values, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=20, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            value = pivot.iloc[i, j]
            label = "" if pd.isna(value) else f"{value:.3f}"
            ax.text(j, i, label, ha="center", va="center", color="white" if not pd.isna(value) and value < 0.5 else "black", fontsize=9)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    fig.tight_layout()
    fig.savefig(OUTDIR / filename)

if not stability_df.empty:
    heatmap(stability_df, "variant", "framing", "label_flip_rate", "Cross-Framing Label Flip Rate vs Neutral", "framing_label_flip_heatmap.png", vmin=0, vmax=1, cmap="magma")
    heatmap(stability_df, "variant", "framing", "mean_abs_score_shift", "Mean Absolute Score Shift vs Neutral", "framing_abs_score_shift_heatmap.png", vmin=0, vmax=max(0.05, stability_df["mean_abs_score_shift"].max()), cmap="magma")

## Boundary Behavior By FEVER Label

**What is computed.** This section uses FEVER labels as factual-status conditions, not as response-level accuracy labels. For NEI, it computes overclaiming: `corrected or partially_corrected`. For REFUTES, it computes correction credit: `corrected or partially_corrected`. For SUPPORTS, it computes over-correction/uncertainty: `not_corrected or uncertain`.

**Why it matters for the paper.** These are construct boundary checks. CALE should reward correction on false claims, avoid unsupported correction on NEI, and avoid penalizing already-supported claims. The heatmaps print exact values in each cell.

In [ ]:
if "reference_label" not in pred_df.columns:
    print("No reference_label found; boundary plots skipped.")
else:
    boundary = pred_df.copy()
    boundary["nei_overclaim"] = np.where(
        boundary["reference_label"] == "NOT ENOUGH INFO",
        boundary["label"].isin(["corrected", "partially_corrected"]).astype(float),
        np.nan,
    )
    boundary["refutes_correction_credit"] = np.where(
        boundary["reference_label"] == "REFUTES",
        boundary["label"].isin(["corrected", "partially_corrected"]).astype(float),
        np.nan,
    )
    boundary["supports_overcorrection"] = np.where(
        boundary["reference_label"] == "SUPPORTS",
        boundary["label"].isin(["not_corrected", "uncertain"]).astype(float),
        np.nan,
    )
    boundary_summary = (
        boundary.groupby(["framing", "variant"], dropna=False)
        .agg(
            nei_overclaim_rate=("nei_overclaim", "mean"),
            refutes_correction_credit=("refutes_correction_credit", "mean"),
            supports_overcorrection_rate=("supports_overcorrection", "mean"),
        )
        .reset_index()
    )
    display(boundary_summary)
    for metric, title, cmap in [
        ("nei_overclaim_rate", "NEI Overclaim Rate by Variant and Framing (lower is better)", "magma"),
        ("refutes_correction_credit", "REFUTES Correction Credit by Variant and Framing", "viridis"),
        ("supports_overcorrection_rate", "SUPPORTS Over-Correction / Uncertainty Rate (lower is better)", "magma"),
    ]:
        heatmap(boundary_summary, "variant", "framing", metric, title, f"boundary_{metric}.png", vmin=0, vmax=1, cmap=cmap)

## CALE Diagnostic Dimensions Across Framing

**What is computed.** This section averages CALE rubric subscores by `dimension × framing`, using the selected CALE variant, usually `full_attack_aware_cale`.

**Why it matters for the paper.** These are CALE-only diagnostic metrics. They should not be used as baseline-vs-CALE outcome metrics because baselines do not output construct subscores. Their value is interpretability: they reveal which part of the construct moves under framing, such as `Correction Accuracy`, `Claim Status Recognition`, `Source Faithfulness`, or `Uncertainty Handling`.

In [ ]:
subscore_rows = []
for row in pred_df.to_dict("records"):
    subscores = row.get("subscores") or {}
    if not isinstance(subscores, dict):
        continue
    for dimension, value in subscores.items():
        subscore_rows.append({
            "id": row.get("id"),
            "model_short": row.get("model_short"),
            "variant": row.get("variant"),
            "framing": row.get("framing"),
            "reference_label": row.get("reference_label"),
            "dimension": dimension,
            "score": value,
        })
subscore_df = pd.DataFrame(subscore_rows)
display(subscore_df.head())

if subscore_df.empty:
    print("No CALE subscores found.")
else:
    chosen_variant = "full_attack_aware_cale" if "full_attack_aware_cale" in set(subscore_df["variant"]) else subscore_df["variant"].iloc[0]
    construct = subscore_df[subscore_df["variant"] == chosen_variant]
    construct_summary = construct.groupby(["dimension", "framing"], dropna=False)["score"].mean().reset_index()
    heatmap(construct_summary, "dimension", "framing", "score", f"CALE Construct Subscores Across Framing ({chosen_variant})", "construct_subscores_by_framing.png", vmin=0, vmax=1)

    bottleneck = (
        construct.groupby(["framing", "dimension"], dropna=False)["score"]
        .mean()
        .reset_index()
        .sort_values(["framing", "score"])
    )
    display(bottleneck.groupby("framing").head(5))
    bottleneck.groupby("framing").head(5).to_csv(OUTDIR / "construct_bottlenecks_by_framing.csv", index=False)

## Model-Level Framing Sensitivity

**What is computed.** For each `model × framing × evaluator variant`, this section computes the model's mean evaluator score.

**Why it matters for the paper.** This shows whether CALE can distinguish target-model response behavior under the same evaluator and same constructed factual conditions. It supports model-discrimination evidence, but should not be framed as a state-of-the-art model ranking.

In [ ]:
model_summary = (
    pred_df.groupby(["model_short", "framing", "variant"], dropna=False)
    .agg(n=("id", "count"), mean_score=("score", "mean"))
    .reset_index()
)
display(model_summary.head())

for variant in ["generic_cale", "attack_aware_cale", "full_attack_aware_cale"]:
    sub = model_summary[model_summary["variant"] == variant]
    if sub.empty:
        continue
    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    for model, mdf in sub.groupby("model_short"):
        mdf = mdf.copy()
        mdf["framing"] = pd.Categorical(mdf["framing"], categories=FRAMING_ORDER, ordered=True)
        mdf = mdf.sort_values("framing")
        ax.plot(mdf["framing"].astype(str), mdf["mean_score"], marker="o", linewidth=2, label=model)
        for _, row in mdf.iterrows():
            ax.annotate(f"{row['mean_score']:.3f}", (str(row["framing"]), row["mean_score"]), textcoords="offset points", xytext=(0, 5), ha="center", fontsize=8)
    ax.set_title(f"Model-Level Mean Score Across Framing ({variant})")
    ax.set_ylabel("mean score")
    ax.set_xlabel("framing")
    ax.set_ylim(*nice_ylim(sub["mean_score"], "mean_score"))
    ax.legend(frameon=True)
    fig.tight_layout()
    fig.savefig(OUTDIR / f"model_framing_mean_score_{variant}.png")

## Qualitative Framing-Flip Cases

**What is computed.** This section lists matched examples where the evaluator label changes between neutral and a non-neutral framing, sorted by absolute score shift.

**Why it matters for the paper.** These cases are candidates for qualitative error analysis. They help explain where a score shift is construct-relevant, such as stronger misinformation pressure, versus construct-irrelevant, such as superficial rhetorical framing.

In [ ]:
if "matched" not in globals() or matched.empty:
    print("No matched framing cases available.")
else:
    cases = matched[matched["label_flip"]].copy()
    cases = cases.sort_values("abs_score_shift", ascending=False)
    display(cases[[c for c in ["id", "model_name", "variant", "framing", "reference_label", "neutral_label", "label", "neutral_score", "score", "score_shift"] if c in cases.columns]].head(30))
    cases.head(100).to_csv(OUTDIR / "qualitative_label_flip_cases.csv", index=False)

## Export Tables

In [ ]:
run_df.to_csv(OUTDIR / "run_inventory.csv", index=False)
metrics_df.to_csv(OUTDIR / "overall_metrics_by_report.csv", index=False)
summary.to_csv(OUTDIR / "shared_metrics_by_framing_variant.csv", index=False)
metric_catalog.to_csv(OUTDIR / "metric_catalog.csv", index=False)
if "stability_df" in globals() and not stability_df.empty:
    stability_df.to_csv(OUTDIR / "cross_framing_stability.csv", index=False)
if "boundary_summary" in globals():
    boundary_summary.to_csv(OUTDIR / "boundary_behavior_by_framing_variant.csv", index=False)
if "subscore_df" in globals() and not subscore_df.empty:
    subscore_df.groupby(["variant", "framing", "dimension"], dropna=False)["score"].agg(["mean", "std", "count"]).reset_index().to_csv(OUTDIR / "construct_subscores_by_framing.csv", index=False)
print("Saved figures and tables to:", OUTDIR)